# 01 — Chuẩn bị đầu vào CNN RGB224


CNN học đặc trưng từ pixel ảnh. Bước này xác nhận split train/dev, mapping sáu mệnh giá,
quy tắc RGB224 và bốn cấu hình sẽ khảo sát. Không đọc ảnh test hoặc cập nhật trọng số mô hình.
Các ô đầu xác định Dataset code/DATA trên Kaggle; kết quả được ghi vào vùng working.


In [ ]:
# KAGGLE_INPUT_SETUP: tìm code và DATA bằng các mức thư mục giới hạn.
from pathlib import Path
import os, sys

kaggle_input = Path('/kaggle/input')
mounts = []
if kaggle_input.is_dir():
    mounts = [p for pattern in ('datasets/*/*', '*')
              for p in kaggle_input.glob(pattern) if p.is_dir()]
project_hint = os.environ.get('CS231_PROJECT_ROOT')
project_candidates = ([Path(project_hint)] if project_hint else
                      [Path.cwd(), *Path.cwd().parents, *mounts,
                       *(child for mount in mounts for child in mount.glob('*') if child.is_dir())])
project_matches = sorted({p.resolve() for p in project_candidates
                          if (p / 'src/CNN/pipeline.py').is_file()})
if len(project_matches) != 1:
    raise FileNotFoundError('Gắn đúng một Dataset code có src/CNN/pipeline.py; '
                            f'có thể đặt CS231_PROJECT_ROOT. Tìm thấy: {project_matches}')
PROJECT_ROOT = project_matches[0]
sys.path.insert(0, str(PROJECT_ROOT))

data_hint = os.environ.get('CS231_DATA_ROOT')
data_candidates = ([Path(data_hint)] if data_hint else
                   [PROJECT_ROOT / 'DATA', *(p / 'DATA' for p in mounts), *mounts])
data_matches = sorted({p.resolve() for p in data_candidates
                       if (p / 'train/10000').is_dir() and (p / 'dev/10000').is_dir()})
if len(data_matches) != 1:
    raise FileNotFoundError('Gắn đúng Dataset DATA có train/dev; '
                            f'có thể đặt CS231_DATA_ROOT. Tìm thấy: {data_matches}')
DATA_ROOT = data_matches[0]
result_hint = os.environ.get('CS231_CNN_RESULTS_ROOT')
RESULTS_ROOT = (Path(result_hint) if result_hint else
                Path('/kaggle/working/CS231_CNN/results_224') if Path('/kaggle/working').is_dir()
                else PROJECT_ROOT / 'src/CNN/results_224').resolve()
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['CS231_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['CS231_DATA_ROOT'] = str(DATA_ROOT)
print('Code:', PROJECT_ROOT)
print('DATA:', DATA_ROOT)
print('Kết quả:', RESULTS_ROOT)


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Image, Markdown
from src.CNN.data import CLASSES, assert_protocol
from src.CNN.runtime import mark_stage
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.is_available())


## 1. Lập manifest và kiểm tra đầu vào

Ảnh được sắp theo mệnh giá tăng dần rồi theo tên file. Mapping chỉ số 0…5 được ghi rõ,
tránh thứ tự từ điển của tên thư mục. Kiểm tra giải mã dùng ảnh đầu của mỗi lớp ở train/dev;
khi huấn luyện, ảnh không đọc được sẽ báo lỗi và không bị bỏ qua âm thầm.


In [ ]:
from src.CNN.pipeline import prepare_inputs
p, counts, architecture = prepare_inputs(DATA_ROOT, RESULTS_ROOT, PROJECT_ROOT)
display(counts.pivot(index='denomination_vnd', columns='split', values='count'))
display(pd.DataFrame({'class_index': range(6), 'denomination_vnd': CLASSES}))
display(pd.read_csv(RESULTS_ROOT / 'inputs/input_checks.csv'))
from src.CNN.data import same_manifest
reference_root = PROJECT_ROOT / 'src/CNN/reference_results'
reference_sources = json.loads((reference_root / 'provenance.json').read_text())
for baseline, source in reference_sources['baselines'].items():
    for split in ('train', 'dev'):
        same_manifest(pd.read_csv(RESULTS_ROOT / 'inputs' / f'{split}_manifest.csv'),
                      pd.read_csv(reference_root / source['folder'] / f'{split}_manifest.csv'))
print('Manifest train/dev khớp baseline KNN và SVM.')


## 2. RGB224 và augmentation trên train

Reader dùng chung đổi BGR→RGB và resize trực tiếp 224×224 bằng OpenCV INTER_LINEAR.
Tensor CNN chia pixel cho 255; không L2 histogram hay dùng mean/std ImageNet. C03/C04
xoay/dịch/scale và đổi sáng/contrast nhẹ trên train; dev/test không augmentation.
Không lật hoặc đổi hue/saturation. Dropout là regularization trên kích hoạt, không phải phép biến đổi ảnh.


In [ ]:
from src.CNN.analysis import augmentation_examples
augmentation_examples(DATA_ROOT, RESULTS_ROOT)
display(Image(filename=str(RESULTS_ROOT / 'inputs/rgb224_examples.png')))
display(Image(filename=str(RESULTS_ROOT / 'inputs/augmentation_examples.png')))


## 3. Kiến trúc, khởi tạo và bốn cấu hình

Một khối gồm hai Conv3×3–BatchNorm–ReLU, rồi MaxPool2×2 và Dropout2d(0,1).
GAP tạo biểu diễn cố định trước Dense128–ReLU–Dropout(0,5)–Dense6 logits.
Ba khối là 32–64–128; mạng sâu thêm khối 256. Conv khởi tạo Kaiming, Dense Xavier;
không nạp pretrained weights. Số tham số/shape bên dưới được tính từ module thực tế.


In [ ]:
from src.CNN.model import build_model
display(pd.DataFrame(p['candidates']))
display(architecture)
display(architecture.groupby('candidate_id', sort=False).parameters.sum().rename('total_parameters'))
print(build_model(p['candidates'][0]))
display(pd.DataFrame([p['training']]))


## 4. Ghi protocol cho chuỗi thực nghiệm

Seed 42, AdamW lr=0,001, weight decay=0,0001, batch64, tối đa25 epoch và patience6
được giữ giữa bốn cấu hình. Chọn checkpoint bằng Macro-F1 dev chưa làm tròn, hòa thì
MAE VND thấp hơn, vẫn hòa giữ epoch sớm. Không chọn theo test hoặc tự gộp train+dev.
Protocol và hash code/manifest cho phép phát hiện thay đổi ở bước sau.


In [ ]:
display(Markdown('Protocol đã ghi tại `inputs/experiment_config.json`.'))
mark_stage(RESULTS_ROOT, '01')
